# Much Ado About Baskets — Exploratory Analysis Notebook (v1 submission)

Original notebook submitted to the **XP Lab** challenge *"Much Ado About Baskets"* (July 2026).
Code and printed outputs are preserved as submitted; charts are not included in this export.

**Context.** "Nearby Basket" is a grocery marketplace. The task: clean messy order data, find product
affinities, and recommend targeted bundle promotions.

**Environment.** This notebook ran inside XP Lab's hosted environment, where the four tables were preloaded
as `df_users`, `df_orders`, `df_order_items` and `df_products`. The dataset belongs to XP Lab and is not
redistributed here.

See the [project README](README.md) for the summary, reviewer feedback and the planned v2.

## 1. Setup and load

In [ ]:
import pandas as pd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

In [ ]:
users = df_users
orders = df_orders
order_items = df_order_items
products = df_products
users.head()

In [ ]:
orders.head()

## 2. Clean product names and parse timestamps

In [ ]:
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'])
orders['order_date'] = orders['order_timestamp'].dt.date
order_items['product_clean'] = order_items['product_raw'].str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
orders.head()

In [ ]:
order_items.head()

In [ ]:
products.head()

In [ ]:
products['product_clean'] = products['product'].str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
products.head()
#str.strip()-remove spaces, str.lower()-lowercase, str.replace(r'\s+', '_', regex=True) -underscore

## 3. Join the four tables into one master table

In [ ]:
unmatched = set(order_items['product_clean']) - set(products['product_clean'])
print(f"Unmatched products in order_items (no match in products table): {len(unmatched)}")
if unmatched:
    print("  Examples:", list(unmatched)[:10])

master = (
    order_items
    .merge(products, on='product_clean', how='left')
    .merge(orders, on='order_id', how='left')
    .merge(users, on='user_id', how='left')
)

Unmatched products in order_items (no match in products table): 0


In [ ]:
master['unit_price_ils'] = pd.to_numeric(master['unit_price_ils'], errors='coerce')
master['quantity'] = pd.to_numeric(master['quantity'], errors='coerce')

## 4. Keep completed orders with valid quantity and price; derive revenue and time fields

In [ ]:
before = len(master)
master = master[
    (master['order_status'] == 'completed') &
    (master['quantity'] > 0) &
    (master['unit_price_ils'] > 0)
].copy()
print(f"Rows before filter: {before:,} → after: {len(master):,}")

master['line_revenue'] = master['quantity'] * master['unit_price_ils']
master['order_dt'] = pd.to_datetime(master['order_timestamp'])
master['order_week'] = master['order_dt'].dt.to_period('W')
master['order_month'] = master['order_dt'].dt.to_period('M')
master['used_promo'] = master['promo_code'].notna() & (master['promo_code'] != 'None')
master.head()

Rows before filter: 2,136 → after: 2,116


In [ ]:
print(f"\nMaster shape: {master.shape}")
print(f"Date range: {master['order_dt'].min().date()} → {master['order_dt'].max().date()}")
print(f"Unique users: {master['user_id'].nunique()}")
print(f"Unique orders: {master['order_id'].nunique()}")
print(f"Unique products: {master['product_clean'].nunique()}")
print(f"Orders with promo: {master.loc[master['used_promo'], 'order_id'].nunique()}")
master.head()

Master shape: (2116, 32)
Date range: 2026-03-01 → 2026-05-31
Unique users: 150
Unique orders: 523
Unique products: 44
Orders with promo: 129


## 5. Basket size, top products and revenue by category

In [ ]:
basket_sizes = master.groupby('order_id')['quantity'].sum()
print(f"Average basket size: {basket_sizes.mean():.1f} items")
print(f"Median basket size:  {basket_sizes.median():.0f} items")
print(f"Std dev:             {basket_sizes.std():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(basket_sizes, bins=range(1, int(basket_sizes.max())+2),
             color='#2ecc71', edgecolor='black', alpha=0.8)
axes[0].set_title('Basket Size Distribution')
axes[0].set_xlabel('Items per Order')
axes[0].set_ylabel('Number of Orders')
axes[0].axvline(basket_sizes.mean(), color='red', linestyle='--', label=f'Mean={basket_sizes.mean():.1f}')
axes[0].legend()

top_products = master.groupby('product_clean')['quantity'].sum().sort_values(ascending=True).tail(10)
axes[1].barh(top_products.index, top_products.values, color='#3498db')
axes[1].set_title('Top 10 Products by Quantity Sold')
axes[1].set_xlabel('Total Units Sold')

plt.tight_layout()
plt.show()

cat_revenue = master.groupby('product_category').agg(
    total_revenue=('line_revenue', 'sum'),
    total_qty=('quantity', 'sum'),
    avg_price=('unit_price_ils', 'mean'),
    unique_orders=('order_id', 'nunique')
).sort_values('total_revenue', ascending=False)

print("\nRevenue by Category:")
print(cat_revenue.round(2))

fresh_split = master.groupby('is_fresh').agg(
    revenue=('line_revenue', 'sum'),
    orders=('order_id', 'nunique'),
    avg_basket_value=('line_revenue', 'mean')
).round(2)
print("\nFresh vs Non-Fresh:")
print(fresh_split)

Average basket size: 5.2 items
Median basket size:  5 items
Std dev:             2.0


Revenue by Category:
                  total_revenue  total_qty  avg_price  unique_orders
product_category
pantry                 12549.36        709      17.54            345
produce                 4377.18        873       5.02            326
dairy                   3971.10        328      12.32            229
meat                    2336.46         65      35.99             47
fruit                   1947.52        231       8.22            134
household               1857.00        129      14.42             93
snacks                  1718.70        123      13.93             59
bakery                  1367.43        118      11.57             87
beverages               1078.42         90      12.14             64
deli                     624.90         45      13.92             32
herbs                    102.01         32       3.22             23

Fresh vs Non-Fresh:
           revenue  orders

## 6. Co-purchase affinity (support and lift)

In [ ]:
transactions = master.groupby('order_id')['product_clean'].apply(lambda x: sorted(set(x)))

multi_item = transactions[transactions.apply(len) >= 2]
print(f"Orders with 2+ distinct products: {len(multi_item)} / {len(transactions)} ({100*len(multi_item)/len(transactions):.0f}%)")

pair_list = []
for items in multi_item:
    pair_list.extend(combinations(items, 2))

pair_counts = pd.Series(pair_list).value_counts().reset_index()
pair_counts.columns = ['pair', 'co_occurrences']

total_orders = master['order_id'].nunique()
pair_counts['support'] = pair_counts['co_occurrences'] / total_orders

prod_freq = master.groupby('product_clean')['order_id'].nunique()
pair_counts['product_a'] = pair_counts['pair'].apply(lambda x: x[0])
pair_counts['product_b'] = pair_counts['pair'].apply(lambda x: x[1])
pair_counts['freq_a'] = pair_counts['product_a'].map(prod_freq) / total_orders
pair_counts['freq_b'] = pair_counts['product_b'].map(prod_freq) / total_orders
pair_counts['lift'] = pair_counts['support'] / (pair_counts['freq_a'] * pair_counts['freq_b'])

print("\n── Top 15 Product Pairs (by co-occurrence count) ──")
print(pair_counts[['pair', 'co_occurrences', 'support', 'lift']].head(15).to_string(index=False))

meaningful = pair_counts[pair_counts['co_occurrences'] >= 5].sort_values('lift', ascending=False)
print("\n── Top 10 Pairs by LIFT (min 5 co-occurrences) ──")
print(meaningful[['pair', 'co_occurrences', 'support', 'lift']].head(10).to_string(index=False))

top_n = 12
top_prods = prod_freq.sort_values(ascending=False).head(top_n).index.tolist()

co_matrix = pd.DataFrame(0, index=top_prods, columns=top_prods)
for _, row in pair_counts.iterrows():
    a, b = row['product_a'], row['product_b']
    if a in top_prods and b in top_prods:
        co_matrix.loc[a, b] = row['co_occurrences']
        co_matrix.loc[b, a] = row['co_occurrences']

plt.figure(figsize=(10, 8))
sns.heatmap(co_matrix, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
plt.title(f'Co-Purchase Heatmap (Top {top_n} Products)')
plt.tight_layout()
plt.show()

Orders with 2+ distinct products: 523 / 523 (100%)

── Top 15 Product Pairs (by co-occurrence count) ──
                      pair  co_occurrences  support     lift
     (tomato, white_onion)             135 0.258126 2.996308
        (cucumber, tomato)              92 0.175908 2.331202
   (cucumber, white_onion)              88 0.168260 2.799513
         (banana, granola)              79 0.151052 6.620253
           (banana, honey)              79 0.151052 6.620253
     (honey, plain_yogurt)              79 0.151052 6.620253
   (granola, plain_yogurt)              79 0.151052 6.620253
          (granola, honey)              79 0.151052 6.620253
    (banana, plain_yogurt)              79 0.151052 6.620253
          (avocado, salsa)              59 0.112811 8.864407
 (avocado, tortilla_chips)              59 0.112811 8.864407
   (salsa, tortilla_chips)              59 0.112811 8.864407
            (eggs, tomato)              52 0.099426 2.325239
       (eggs, white_onion)              48

## 7. Trends over time: revenue, orders, day of week, promo usage

In [ ]:
weekly = master.groupby(master['order_dt'].dt.to_period('W')).agg(
    revenue=('line_revenue', 'sum'),
    orders=('order_id', 'nunique'),
    avg_basket_value=('line_revenue', lambda x: x.sum()),
    items_sold=('quantity', 'sum')
)
weekly.index = weekly.index.to_timestamp()
weekly['avg_order_value'] = weekly['revenue'] / weekly['orders']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(weekly.index, weekly['revenue'], 'g-o', markersize=4, label='Weekly Revenue (ILS)')
axes[0].set_ylabel('Revenue (ILS)')
axes[0].set_title('Weekly Revenue & Order Volume')
axes[0].legend(loc='upper left')
ax0b = axes[0].twinx()
ax0b.plot(weekly.index, weekly['orders'], 'b--x', markersize=4, label='Orders', alpha=0.7)
ax0b.set_ylabel('Orders')
ax0b.legend(loc='upper right')

axes[1].plot(weekly.index, weekly['avg_order_value'], 'r-o', markersize=4, label='Avg Order Value (ILS)')
axes[1].set_ylabel('ILS')
axes[1].set_title('Average Order Value Over Time')
axes[1].legend()
axes[1].set_xlabel('Week')

plt.tight_layout()
plt.show()

master['dow'] = master['order_dt'].dt.day_name()
dow_order = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
dow_stats = master.groupby('dow')['order_id'].nunique().reindex(dow_order)

plt.figure(figsize=(8, 4))
plt.bar(dow_stats.index, dow_stats.values, color='#9b59b6')
plt.title('Orders by Day of Week')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

promo_weekly = master.groupby(master['order_dt'].dt.to_period('W')).agg(
    promo_orders=('used_promo', 'sum'),
    total_items=('order_item_id', 'count')
)
promo_weekly.index = promo_weekly.index.to_timestamp()
promo_weekly['promo_pct'] = promo_weekly['promo_orders'] / promo_weekly['total_items'] * 100

plt.figure(figsize=(12, 4))
plt.plot(promo_weekly.index, promo_weekly['promo_pct'], 'o-', color='#e67e22')
plt.title('Promo Code Usage Rate Over Time (%)')
plt.ylabel('% of Line Items with Promo')
plt.xlabel('Week')
plt.tight_layout()
plt.show()

## 8. RFM customer segmentation

In [ ]:
snapshot_date = master['order_dt'].max() + pd.Timedelta(days=1)

rfm = master.groupby('user_id').agg(
    recency=('order_dt', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('line_revenue', 'sum')
).round(2)

rfm['r_score'] = pd.cut(rfm['recency'].rank(method='first'), bins=4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.cut(rfm['frequency'].rank(method='first'), bins=4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.cut(rfm['monetary'].rank(method='first'), bins=4, labels=[1,2,3,4]).astype(int)

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

def segment_label(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 3 and f >= 3: return 'Champions'
    if r >= 3 and f >= 2: return 'Loyal'
    if r >= 3 and f == 1: return 'New Customers'
    if r == 2 and f >= 2: return 'At Risk'
    if r <= 1 and f >= 2: return 'Hibernating'
    if r <= 1 and f == 1: return 'Lost'
    return 'Other'

rfm['segment'] = rfm.apply(segment_label, axis=1)

seg_summary = rfm.groupby('segment').agg(
    count=('recency', 'size'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).sort_values('avg_monetary', ascending=False).round(1)

print("Customer Segments:")
print(seg_summary)
print(f"\nTotal customers: {len(rfm)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = rfm['segment'].value_counts()
axes[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.0f%%', startangle=140)
axes[0].set_title('Customer Segments (Count)')

seg_rev = rfm.groupby('segment')['monetary'].sum().sort_values(ascending=True)
axes[1].barh(seg_rev.index, seg_rev.values, color='#2ecc71')
axes[1].set_title('Total Revenue by Segment')
axes[1].set_xlabel('Revenue (ILS)')

plt.tight_layout()
plt.show()

user_promo = master.groupby('user_id')['used_promo'].mean()
rfm['promo_rate'] = rfm.index.map(user_promo)

promo_by_seg = rfm.groupby('segment')['promo_rate'].mean().sort_values(ascending=False)
print("\nPromo Usage Rate by Segment:")
print(promo_by_seg.round(3))

Customer Segments:
               count  avg_recency  avg_frequency  avg_monetary
segment
Lost               9         34.1            3.0         227.6
New Customers     18         10.3            3.0         222.8
Hibernating       29         35.7            3.4         216.4
Champions         39          8.2            4.1         216.3
At Risk           26         21.0            3.7         205.8
Loyal             18          7.8            3.0         203.2
Other             11         21.2            3.0         195.7

Total customers: 150

Promo Usage Rate by Segment:
segment
Hibernating      0.328
At Risk          0.299
Champions        0.271
Lost             0.250
Other            0.246
New Customers    0.229
Loyal            0.207
Name: promo_rate, dtype: float64


## 9. Promotions, regions and devices

In [ ]:
orders['discount_ils'] = pd.to_numeric(orders['discount_ils'], errors='coerce')
orders['order_value_ils'] = pd.to_numeric(orders['order_value_ils'], errors='coerce')

promo_orders = orders[orders['promo_code'].notna() & (orders['promo_code'] != 'None')].copy()
print(f"Orders with promo code: {len(promo_orders)} / {len(orders)} ({100*len(promo_orders)/len(orders):.1f}%)")

if len(promo_orders) > 0:
    promo_summary = promo_orders.groupby('promo_code').agg(
        uses=('order_id', 'count'),
        avg_discount=('discount_ils', 'mean'),
        avg_order_value=('order_value_ils', 'mean'),
        total_discount=('discount_ils', 'sum')
    ).sort_values('uses', ascending=False).round(2)
    print("\nPromo Code Performance:")
    print(promo_summary)

    order_level = master.groupby('order_id').agg(
        basket_size=('quantity', 'sum'),
        revenue=('line_revenue', 'sum'),
        used_promo=('used_promo', 'first')
    )

    comparison = order_level.groupby('used_promo').agg(
        count=('revenue', 'size'),
        avg_basket_size=('basket_size', 'mean'),
        avg_revenue=('revenue', 'mean')
    ).round(2)
    comparison.index = ['No Promo', 'With Promo']
    print("\nPromo vs No-Promo Baskets:")
    print(comparison)

    avg_disc = promo_orders['discount_ils'].mean()
    avg_val = promo_orders['order_value_ils'].mean()
    print(f"\nAvg discount: {avg_disc:.2f} ILS on avg order of {avg_val:.2f} ILS = {100*avg_disc/avg_val:.1f}% discount rate")

region_stats = master.groupby('delivery_region_raw').agg(
    orders=('order_id', 'nunique'),
    revenue=('line_revenue', 'sum'),
    users=('user_id', 'nunique')
)
region_stats['avg_order_value'] = region_stats['revenue'] / region_stats['orders']
region_stats['orders_per_user'] = region_stats['orders'] / region_stats['users']
print("\nPerformance by Region:")
print(region_stats.sort_values('revenue', ascending=False).round(2))

device_stats = master.groupby('device_type').agg(
    orders=('order_id', 'nunique'),
    revenue=('line_revenue', 'sum')
)
device_stats['avg_order_value'] = device_stats['revenue'] / device_stats['orders']
print("\nPerformance by Device:")
print(device_stats.sort_values('revenue', ascending=False).round(2))

Orders with promo code: 137 / 531 (25.8%)

Promo Code Performance:
             uses  avg_discount  avg_order_value  total_discount
promo_code
FRESHAPRIL     54          7.93            40.54          427.96
WEEKENDHOST    32          9.91            83.98          317.11
BREAKFAST10    30          5.87            93.64          176.07
STAPLES5       13          5.88            69.62           76.50
TEST            8          0.00            32.02            0.00

Promo vs No-Promo Baskets:
            count  avg_basket_size  avg_revenue
No Promo      394             4.96        59.13
With Promo    129             6.11        66.92

Avg discount: 7.28 ILS on avg order of 64.58 ILS = 11.3% discount rate

Performance by Region:
                     orders  revenue  users  avg_order_value  orders_per_user
delivery_region_raw
center                  100  6090.57     49            60.91             2.04
south                    84  5507.21     47            65.56             1.79
coast     

## 10. Exploratory analysis memo

The memo below is printed by the notebook. Section C is computed from the data; in the submitted run it read:

- Avg basket size: 5.2 items (median: 5)
- Strongest product affinity (lift): ('grapes', 'hummus') — lift = 13.07, seen 11 times
- Top 20% of customers drive 30% of revenue
- Customer segments: Champions 39, Hibernating 29, At Risk 26, New Customers 18, Loyal 18, Other 11, Lost 9

In [ ]:
print("""
============================================================
     EXPLORATORY ANALYSIS MEMO - "Nearby Basket"
     Bundle Promotion Planning Readout
============================================================

=== A. DATA QUALITY NOTES ===

1. The orders table uses 'order_timestamp', not 'order_date' - naive code
   that references 'order_date' crashes immediately (KeyError).
2. order_items joins to products via 'product_raw' -> 'product' (after
   cleaning), NOT via any 'product_id' foreign key.
3. Non-completed orders (cancelled, returned) must be filtered out
   BEFORE computing revenue or basket metrics - the template code
   doesn't do this, inflating volume numbers.
4. There is a minor change in location names, for example: Jerusalem and JERUSALEM.
5. The recency does not match the frequency - therefore there is an error in the data that must be rechecked in the source tables.

=== B. TEMPTING SHORTCUTS -> WRONG PROMOTIONS ===

TRAP 1: "Bundle the most-sold products together"
  The top-selling products appear in many baskets simply because they're
  popular (staples like milk, eggs, bread). High co-occurrence != affinity.
  A bundle of two staples won't increase basket size - customers already
  buy both. Use LIFT, not raw frequency, to find genuine affinity.

TRAP 2: "Give discounts to our best customers"
  Champions already buy frequently at full price. Discounting to them
  burns margin without changing behavior. Promos should target segments
  where behavior CAN change: New Customers (to build habit), At Risk
  (to prevent churn), or Hibernating (to reactivate).

TRAP 3: "The promo worked - average order value went up"
  If promo users already had higher baskets before the promo, the lift is
  selection bias, not causal effect. Compare pre/post for the SAME users,
  or compare promo-eligible vs. non-eligible cohorts.

TRAP 4: "Fresh + non-fresh bundles double the basket"
  Fresh items have short shelf life and high return rates. Bundling fresh
  with non-fresh may increase spoilage costs. Analyze whether fresh items
  in promo baskets have higher wastage.

TRAP 5: "One region outperforms - roll the promo nationally"
  Regional differences (demographics, competition, delivery costs) mean a
  promo that works in 'center' may fail in 'north'. Test per-region.
""")

print("=== C. KEY FINDINGS (from this dataset) ===\n")

bs = master.groupby('order_id')['quantity'].sum()
print(f"  Avg basket size: {bs.mean():.1f} items (median: {bs.median():.0f})")

if len(meaningful) > 0:
    top = meaningful.iloc[0]
    print(f"  Strongest product affinity (lift): {top['pair']} - lift={top['lift']:.2f}, seen {top['co_occurrences']} times")

user_rev = master.groupby('user_id')['line_revenue'].sum().sort_values(ascending=False)
top20_pct = user_rev.head(int(len(user_rev)*0.2)).sum() / user_rev.sum() * 100
print(f"  Top 20% of customers drive {top20_pct:.0f}% of revenue")

print(f"  Customer segments: {rfm['segment'].value_counts().to_dict()}")

print("""

=== D. RECOMMENDED PROMOTION TESTS ===

Based on the analysis, we recommend testing these 4 promotions:

1. HIGH-LIFT BUNDLE (Data-Driven Affinity)
   Bundle the top pair by LIFT (not raw frequency).
   Target: All customers. Measure: incremental basket size vs. control.

2. REACTIVATION OFFER for Hibernating Segment
   10-15% discount on next order for users in the 'Hibernating' segment.
   Target: Hibernating users only. Measure: reactivation rate within 14 days.

3. NEW CUSTOMER SECOND-ORDER NUDGE
   Free delivery or small discount on 2nd order placed within 7 days of 1st.
   Target: 'New Customers' segment. Measure: 2nd-order conversion rate.

4. CROSS-CATEGORY BUNDLE (Fresh + Pantry)
   Bundle a popular fresh item with a complementary pantry item
   (e.g., if tomato + pasta show high lift).
   Target: Users who buy only one category. Measure: category expansion rate.

   Monitor fresh-item spoilage/return rate during the test.

=== E. WHAT NOT TO DO ===

X Don't blanket-discount staples (milk, bread, eggs) - no incremental lift.
X Don't reward Champions with discounts - they already buy at full price.
X Don't evaluate promo success by comparing promo-users to non-users
  without controlling for self-selection.
X Don't assume one region's result generalizes nationally.
""")